Récupération flux vidéo dans un naviguateur:

In [ ]:
# --- IMPORTATION DES BIBLIOTHEQUES ---
# traitement d'images et calculs
import cv2                                                                      # OpenCV pour le traitement d'image
import numpy as np                                                              # pour les calculs avec les matrices d'images

# système et gestion de données
import time                                                                     # pour la gestion du temps (horodatage et pauses entre les prises d'image)
import os                                                                       # pour interagir avec le système (gestion des fichiers, dossiers et configuration du fuseau horaire)
import pickle                                                                   # sert à sérialiser et désérialiser des objets Python, cad transformer une structure de données complexe (liste, dictionnaire, modèle IA) en un flux d'octets

# interface et capture google colab
from IPython.display import display, Javascript, clear_output                   # interface Google Colab
from base64 import b64decode                                                    # pour décoder les images envoyées par le navigateur
from google.colab.output import eval_js                                         # pour exécuter du JavaScript dans Colab
from google.colab.patches import cv2_imshow                                     # version spécifique d'OpenCV pour afficher des images dans Colab

# types pour le type hinting
from typing import Optional, Tuple                                              # optional pour indiquer qu'une valeur peut être d'un type donné ou Non

# --- INSTALLATION CONDITIONNELLE ---
# IA pour reconnaissance
try:
  import face_recognition                                                       # utilisé pour la détection et la reconnaissance faciale
  print("face_recognition est déjà installé")
except ImportError:
  print("installation de face_recognition...")
  !pip install face_recognition
  import face_recognition

In [ ]:
# --- FONCTION DE CAPTURE PHOTO (Interface JavaScript) ---
def take_photo_auto(quality: float = 1.0)-> np.ndarray:
  """
  Lance la webcam via JavaScript, capture une image fixe et la convertit en format exploitable par OpenCV

  Etapes:
  - Exécution de JS pour accéder au périphérique 'user' (selfie)
  - Capture du flux dans un canvas HTML5
  - Décodage de la chaîne Base64 en tableau NumPy (BGR)

  Args:
      quality (float): Qualité de compression JPEG (de 0.0 à 1.0) avec 1.0 comme valeur par défaut pour une précision maximale en reconnaissance faciale

  Returns:
      np.ndarray: Image au format OpenCV (matrice multidimensionnelle BGR)
                  Renvoie None ou lève une erreur si l'accès à la caméra est refusé
  """
  js = Javascript('''                                                           // stocke une chaîne de caractères contenant du code JS exécuté par le navigateur de l'utilisateur
  async function takePhotoAuto(quality) {                                       // déclare une fonction asynchrone en JS (nécessaire pour attendre l'activation de la caméra)
    const video = document.createElement('video');                              // crée un élément HTML <video> invisible pour recevoir le flux de la caméra
    video.style.display = 'none';                                               // cache l'élément vidéo pour qu'il n'apparaisse pas sur la page Colab
    // const stream = await navigator.mediaDevices.getUserMedia({video: true}); // demande au navigateur l'autorisation d'accéder à la webcam
    const stream = await navigator.mediaDevices.getUserMedia({video: { facingMode: "user" }}); // demande au navigateur l'autorisation d'accéder à la webcam, (facingMode: "user" et non "environment") force l'utilisation de la caméra de selfie pour éviter les mauvaises utilisations
    document.body.appendChild(video);                                           // ajoute temporairement la vidéo au document pour permettre l'initialisation du flux vidéo
    video.srcObject = stream;                                                   // connecte le flux de la caméra à l'élément vidéo
    await video.play();                                                         // démarre la lecture de la vidéo
    await new Promise(resolve => setTimeout(resolve, 250));                     // attend 250ms pour laisser le temps à l'autofocus et à l'exposition de se stabiliser
    const canvas = document.createElement('canvas');                            // crée un élément <canvas> (zone de dessin) pour transformer la vidéo en image fixe
    canvas.width = video.videoWidth; canvas.height = video.videoHeight;         // définit la largeur et la hauteur du canvas égale à celle du flux vidéo reçu
    canvas.getContext('2d').drawImage(video, 0, 0);                             // prend la photo en dessinant l'image actuelle de la vidéo sur le canvas
    stream.getVideoTracks()[0].stop();                                          // éteint la webcam
    video.remove();                                                             // suppression de l'élément vidéo temporaire du code HTML de la page
    return canvas.toDataURL('image/jpeg', quality);                             // renvoie l'image sous forme de texte encodé en Base64
  }
  ''')
  display(js)                                                                   # envoie le code JS au navigateur pour qu'il soit interprété
  data = eval_js('takePhotoAuto({})'.format(quality))                           # exécute la fonction JS et récupère la chaîne de caractères (Base64) dans la variable Python data
  binary = b64decode(data.split(',')[1])                                        # conversion du texte Base64 contenant les données de l'image reçue pour retrouver les données binaires brutes, un format exploitable par OpenCV (tableau Numpy)
  nparr = np.frombuffer(binary, np.uint8)                                       # transforme ces données binaires en un tableau de nombres (vecteur de pixels) via NumPy
  return cv2.imdecode(nparr, cv2.IMREAD_COLOR)

In [ ]:
# --- FONCTION D'ATTENTE AUTORISATION ACCES CAMERA ---
def wait_for_camera() -> None:
  """
  Bloque l'exécution du script Python jusqu'à ce que l'utilisateur valide l'accès à la webcam.

  Etapes:
  - Affichage d'un message d'état initial dans la console
  - Injection du script dans le navigateur via la fonction 'display'
  - Appel de la fonction JavaScript 'askPermission' :
     - Ouvre la fenêtre contextuelle (pop-up) de demande de permission du navigateur
     - En cas de succès, stoppe immédiatement la récupération du flux vidéo pour libérer le périphérique
  - Récupération du statut (accordé/refusé) pour synchroniser l'état du navigateur avec Python
  - Affichage du statut final

  Returns:
      None: La fonction ne renvoie rien
  """
  print("en attente de l'autorisation de la caméra ...")
  js_auth = Javascript('''
      async function askPermission() {                                          // demande de l'autorisation d'accès au flux vidéo
          try {
              const stream = await navigator.mediaDevices.getUserMedia({video: true});
              stream.getTracks().forEach(track => track.stop());                // une fois l'accès obtenu, on coupe immédiatement le flux vidéo
              return "granted";                                                 // accès accordé
          } catch (err) {
              return "denied";                                                  // accès refusé par l'utilisateur ou erreur technique
          }
      }
  ''')
  display(js_auth)                                                              # exécution de la fonction JS 'js_auth'
  status = eval_js('askPermission()')                                           # exécution de la fonction JS 'askPermission()' et récupèration de son résultat dans une variable en Python
                                                                                # analyse du résultat renvoyé par le navigateur
  if status == "granted":
       print("accès caméra validé")
  else:
      print("veuillez accepter l'accès à la caméra dans votre navigateur ...")  # message d'erreur

In [ ]:
# --- FONCTION LISTANT TOUTES LES CAMERAS DETECTEES (utile sur téléphone en particulier) ---
def check_cameras() -> str:
  """
  Interroge le navigateur pour obtenir la liste de tous les périphériques d'entrée vidéo disponibles
  (Cette fonction est utile pour les téléphones portables qui ont plusieurs caméras (avant/arrière))

  Etapes:
  - Force une demande `getUserMedia` pour réveiller les permissions
  - Utilise `enumerateDevices()` pour scanner le matériel connecté
  - Filtre les résultats pour ne conserver que les types 'videoinput'
  - Extrait le label (nom) et une version raccourcie de l'ID unique du périphérique

  Returns:
      str: Une chaîne de caractères formatée listant les caméras trouvées
  """
  js = Javascript('''
  async function listDevices() {
    await navigator.mediaDevices.getUserMedia({video: true});
    const devices = await navigator.mediaDevices.enumerateDevices();            // récupère la liste complète des périphériques multimédias connectés (micros, caméras, ...)
    const videoDevices = devices.filter(device => device.kind === 'videoinput');// filtre la liste pour ne garder que les entrées vidéo (élimine les microphones)
    let info = "caméras trouvées :\\n";                                         // initialise une chaîne de caractères pour stocker les informations à afficher
    videoDevices.forEach((device, index) => {                                   // parcourt chaque caméra trouvée pour construire le message informatif
      info += `${index}: ${device.label || 'caméra inconnue'} (ID (5st numbers): ${device.deviceId.substring(0,5)})\\n`; // ajoute l'index, le nom de la caméra (ou 'inconnue') et les 5 premiers caractères de son ID unique
    });
    return info;
  }
  ''')
  display(js)
  return eval_js('listDevices()')

# print(check_cameras())

In [ ]:
# --- PARAMETRES MODIFIABLES ---
SAVE_IMAGE = 0                                                                  # variable pour choix d'enregistrement ou non de l'image prise (1 = enregistrer et 0 = ne pas enregistrer)
NB_MODE = True                                                                  # variable pour choisir la palette de couleur de l'image, True pour Noir & Blanc, False pour Couleur
TRAIN_WIDTH = 180                                                               # largeur de la preview del'image d'entrainement en pixels
TRAIN_HEIGHT = 135                                                              # hauteur de la preview del'image d'entrainement en pixels
WIDTH = 48                                                                      # largeur de l'image finale crop autour du visage en pixels
HEIGHT = 48                                                                     # hauteur de l'image finale crop autour du visage en pixels
padding = 40                                                                    # marge de découpe autour du visage en pixels
CONFIDENCE_THRESHOLD = 0.60                                                     # seuil minimum de certitude (de 0.0 à 1.0) pour valider la détection d'un visage
EPSILON = 1e-6                                                                  # valeur de sécurité anti division par zéro
DELAY = 5                                                                       # temps d'attente entre deux captures d'images en secondes
photos_needed = 3                                                               # nombre de photos nécessaires à l'entrainement pour créer un profil fiable
thickness = 3 - 1                                                               # épaisseur ajustée pour le dessin du cadre, OpenCV centre la ligne sur le tracé, on réduit donc de 1 pour que le bandeau s'aligne parfaitement sur le bord extérieur du cadre
DB_FILE = "face_signature.pkl"                                                  # nom du fichier de sauvegarde des signatures des visages encodées
mean_encoding = None                                                            # signature la valeur moyenne du visage initialisée à None
print("initialisation terminée")

In [ ]:
# --- CONFIGURATION INITIALE ET ENRÔLEMENT DE LA SIGNATURE FACIALE ---
def setup_recognition() -> tuple[bool, Optional[np.ndarray]]:
  """
  Orchestre la configuration initiale du système et l'enrôlement de l'utilisateur

  Etapes:
  - Validation matérielle : Attente de l'activation de la caméra
  - Choix utilisateur, propose l'activation ou non de la reconnaissance faciale
      - Si l'utilisateur répond "y" à l'activation, alors le mode Reconnaissance est activé
      - Sinon (réponse "n" ou autre), alors le mode Détection Simple (Anonyme) est activé et la fonction s'arrête ici en renvoyant False
  - Tentative de chargement d'une signature existante (.pkl) (si mode Reconnaissance)
      - Si un fichier de signature (.pkl) existe déjà sur le disque
        - Alors l'utilisateur choisit s'il veut l'utiliser ou non
          - Si l'utilisateur accepte et que le fichier n'est pas corrompu
            - Alors la signature est chargée et on saute l'étape d'entraînement
            - Sinon (refus ou fichier corrompu), alors on définit la signature comme vide (None) pour forcer un nouvel entraînement
      - Sinon (pas de fichier trouvé), alors on se prépare directement pour la phase d'entraînement
  - Phase d'entraînement (si signature vide et si mode Reconnaissance)
    - Tant que le nombre de photos valides est inférieur au quota requis
      - Alors on tente une capture d'image
        - Si aucun visage n'est détecté ou si plus d'un visage est présent
            - Alors on affiche une erreur et on recommence la boucle (on n'enregistre rien)
        - Sinon (exactement un visage détecté)
            - Alors on extrait la signature faciale (les encodages faciaux via 'face_recognition', 128 dimensions) et on l'ajoute à la liste
    - Quand les captures sont terminées, on calcule la moyenne mathématique des signatures pour créer un profil stable
      - Si l'utilisateur souhaite la sauvegarder
        - Alors on écrit la signature sur le disque
      - Sinon:
        - Alors la signature ne sera conservée qu'en mémoire vive pour la session en cours

  Global Variables:
      mean_encoding (np.ndarray): Met à jour la signature de référence au niveau global

  Returns:
      tuple:
          - bool: État de l'activation (True si reconnaissance faciale activée)
          - np.ndarray ou None: Le vecteur d'encodage facial de 128 dimensions
  """
  # --- VARIABLE GLOBALE ---
  global mean_encoding                                                          # permet de modifier la signature faciale au niveau global pour qu'elle soit accessible par les autres fonctions et éviter de la redéfinir dans la fonction (mean_encoding = None)

  # --- INITIALISATION LOCALE ---
  mean_encoding = None
  facial_recognition_enabled = False

  # --- VERIFICATION CAMERA ---
  wait_for_camera()
  print("système de capture d'image prêt à être lancé")
  time.sleep(2)
  clear_output(wait=False)                                                      # nettoie immédiatement la console en effaçant le message précédent pour libérer l'affichage avant la suite

  # --- OPTOUT OPTION ---
  user_choice = input("souhaitez vous activer la reconnaissance faciale afin d'améliorer les performances? \n(y/n) : ").lower() # récupère le choix utilisateur et force la mise en minuscules pour la comparaison
  facial_recognition_enabled = True if user_choice == 'y' else False            # définit le mode de fonctionnement comme booléen, True si 'y', sinon False (par défaut)
  if not facial_recognition_enabled:
    print("\n--- MODE DÉTECTION SIMPLE ACTIVÉ (Anonyme) ---")
  else:
    print("\n--- MODE RECONNAISSANCE ACTIVÉ ---")

    # --- VERIFICATION BDD EXISTANTE ---
    if os.path.exists(DB_FILE):
      use_saved = input("une signature de visage existe déjà, voulez vous l'utiliser? \n(y/n) : ").lower()  # récupère le choix utilisateur et force la mise en minuscules pour la comparaison
      if use_saved == 'y':
        try:                                                                    # vérification si corruption ou absence du fichier Pickle
          with open(DB_FILE, 'rb') as f:
            mean_encoding = pickle.load(f)
          print("signature chargée avec succès")
        except (EOFError, pickle.UnpicklingError):
          mean_encoding = None
          print("erreur le fichier de signature semble corrompu, démarrage d'un nouvel entraînement")
      else:
        mean_encoding = None
        print("refus d'utiliser la signature existante, préparation d'un nouvel entraînement ...")

    # --- ENTRAINEMENT (si pas de signature chargée) ---
    if mean_encoding is None:
      print("--- PHASE D'ENTRAINEMENT ---")
      known_encodings = []                                                      # initialise une liste vide pour stocker temporairement des vecteurs de 128 dimensionsles ("signatures" des visages capturés)
      while len(known_encodings) < photos_needed:                               # boucle pour prendre la ou les photo(s) nécessaire(s), utilisation de while pour forcer la réussite de l'enrôlement
        print(f"\ncapture {len(known_encodings) + 1}/{photos_needed} : veuillez regarder la caméra ...")
        frame = take_photo_auto()                                               # prise d'une photo via la caméra
        if frame is None:
          print("erreur, caméra indisponible, nouvel essai ...")
          time.sleep(1)
          continue
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)                      # convertit l'image de BGR (OpenCV) vers RGB (format requis par face_recognition)
        encodings = face_recognition.face_encodings(rgb_frame)                  # encodage: analyse l'image pour extraire les 128 points caractéristiques du visage et générer un vecteur de 128 dimensions (descripteur facial) unique au visage
        num_detected = len(encodings)                                           # calcul du nombre de visages détectés
        if num_detected == 0:
          print("aucun visage détecté, réessayez")
          continue
        elif num_detected > 1:
          print(f"{num_detected} visages détectés, l'entrainement nécessite UNE SEULE personne dans le champ de vision de la caméra")
          continue                                                              # on ne fait rien, la boucle recommence sans ajouter de signature
        else:                                                                   # si un seul visage détecté
          known_encodings.append(encodings[0])                                  # ajoute la signature du premier visage détecté à notre liste de référence
          print("visage capturé avec succès")
          cv2_imshow(cv2.resize(frame, (TRAIN_WIDTH, TRAIN_HEIGHT)))            # affiche l'image capturée pour l'entrainement si un visage a été détecté
        time.sleep(2)
      if not known_encodings:
        raise Exception("enrôlement échoué, aucun visage n'a été trouvé")
      mean_encoding = np.mean(known_encodings, axis=0)                          # calcule la moyenne (centroïde) de chaque dimension du vecteur d'encodage (128 points) du ou des signature(s) (indispensable pour NumPy) pour créer une signature de référence stable (un profil optimal)

      # --- SAUVEGARDE DANS LA BDD ---
      save_choice = input("voulez vous enregistrer ce visage pour la prochaine fois? \n(y/n) : ").lower()  # récupère le choix utilisateur et force la mise en minuscules pour la comparaison
      if save_choice == 'y':
        with open(DB_FILE, 'wb') as f:
          pickle.dump(mean_encoding, f)
        print(f"signature enregistrée dans {DB_FILE}")
  return facial_recognition_enabled, mean_encoding

Modèle estimation des émotions:

In [ ]:
# --- IMPORTATION D'UNE BIBLIOTHEQUE ---
import os                                                                       # pour interagir avec le système (gestion des fichiers, dossiers et configuration du fuseau horaire)

# --- PARAMETRES ---
repo_url = "https://github.com/arthur5775/CS_Projet_DOMINANCE.git"
folder_name = "CS_Projet_DOMINANCE"

# --- CLONAGE CONDITIONNEL ---
if not os.path.exists(folder_name):
  print(f"clonage du dépôt {folder_name}...")
  !git clone {repo_url}                                                         # clonage du dépôt GitHub du projet dans l'environnement actuel
else:
  print(f"le dossier {folder_name} existe déjà. Pas besoin de le cloner")

In [ ]:
%cd ./CS_Projet_DOMINANCE
                                                                                # passage dans le répertoire du projet

In [ ]:
# --- IMPORTATION DES BIBLIOTHÈQUES ---
# configuration de l'IA
import torch                                                                    # framework principal pour le Deep Learning et le calcul tensoriel
import torchvision.models as models                                             # pour accéder aux architectures de modèles préentraînés
import torchvision.transforms as transforms                                     # outils de prétraitement des images (redimensionnement et normalisation)
import torch.nn as nn                                                           # modules pour construire les couches des réseaux de neurones
from torch.autograd import Variable                                             # gestion du calcul des gradients

# manipulation d'images et visualisation
from PIL import Image                                                           # sert à ouvrir et manipuler des fichiers images
import matplotlib.pyplot as plt                                                 # pour générer des graphiques
import numpy as np                                                              # pour les calculs avec les matrices d'images

# modèle personnalisé
from models.resnet_reg2 import (
    ResNet50RegressionThreeOutputs,
)

In [ ]:
# --- PARAMETRES DE NORMALISATION IMAGES ---
CAER_S_MEAN = [0.5078952312469482, 0.5078952312469482, 0.5078952312469482]
CAER_S_STD = [0.2549697160720825, 0.2549697160720825, 0.2549697160720825]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# --- PARAMÈTRES DE LABELS (VAD) ---
CAER_S_LABEL_MEAN = [-0.31888335943222046, 0.46993690729141235, 0.09596027433872223]
CAER_S_LABEL_STD = [1.3228799104690552, 1.2825294733047485, 1.3567899465560913]
IMAGENET_LABEL_MEAN = [0.0, 0.0, 0.0]
IMAGENET_LABEL_STD = [1.0, 1.0, 1.0]

In [ ]:
"""
Activer le GPU dans Colab:
- dans le menu Exécution (en haut à gauche)
- modifier le type d'exécution
- Sous "Accélérateur matériel", choisir T4 GPU
- enregistrer.
"""

# --- CONFIGURATION DU DEVICE ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device : {device}")

# --- CHARGEMENT DU MODELE ---
model_instance = ResNet50RegressionThreeOutputs()

# --- CHEMIN DES POIDS ---
weights_path = "./runs/Caer-S_ResNet50/best_model_state.pth"

# --- CHARGEMENT DES POIDS ---
loaded_data = torch.load(weights_path, map_location=device)

if isinstance(loaded_data, dict) and "model" in loaded_data:
    state_dict = loaded_data["model"]
else:
    state_dict = loaded_data

model_instance.load_state_dict(state_dict)
model_instance.to(device)
model_instance.eval()
model_VAD = model_instance

# --- PRÉPARATION DES TRANSFORMATIONS ---
def update_transform_weights(new_mean: list, new_std: list) -> None:
    """
    Initialise ou met à jour l'objet 'transform' avec de nouveaux poids de normalisation

    Args:
        new_mean (list): Liste de 3 valeurs pour la moyenne (R, G, B)
        new_std (list): Liste de 3 valeurs pour l'écart-type (R, G, B)

    Returns:
      None: La fonction ne renvoie rien
    """
    global transform
    new_transform = transforms.Compose([
        transforms.Resize((48, 48)),                                            # redimensionnement de l'image en 48x48 pixels
        transforms.ToTensor(),                                                  # conversion de l'image en tenseur PyTorch
        transforms.Normalize(mean=new_mean, std=new_std)                        # normalisation des couleurs (moyenne et écart-type)
    ])
    transform = new_transform

def update_normalization_by_path(path: str) -> None:
    """
    Synchronise les paramètres de pré-traitement (images) et de post-traitement (VAD) en fonction du dataset détecté dans le chemin des poids

    Global Variables:
      - transform (torchvision.transforms): La chaîne de transformation d'image
      - label_mean_vad (torch.Tensor): La moyenne pour la dé-normalisation des scores
      - label_std_vad (torch.Tensor): L'écart-type pour la dé-normalisation des scores

    Returns:
      None: La fonction ne renvoie rien
    """
    global label_mean_vad, label_std_vad

    if "Caer-S" in path:
        update_transform_weights(CAER_S_MEAN, CAER_S_STD)
        label_mean_vad = torch.tensor(CAER_S_LABEL_MEAN).to(device)
        label_std_vad = torch.tensor(CAER_S_LABEL_STD).to(device)
    else:
        update_transform_weights(IMAGENET_MEAN, IMAGENET_STD)
        label_mean_vad = torch.tensor(IMAGENET_LABEL_MEAN).to(device)
        label_std_vad = torch.tensor(IMAGENET_LABEL_STD).to(device)

update_normalization_by_path(weights_path)
# print(transform)
# print(label_mean_vad)
# print(label_std_vad)

code export des données collectées en csv:

In [ ]:
# --- IMPORTATION DES BIBLIOTHEQUES ---
import pandas as pd                                                             # pour manipuler et analyser des données structurées comme des fichiers CSV
from pathlib import Path                                                        # pour manipuler les chemins des fichiers
import os                                                                       # pour interagir avec le système (gestion des fichiers, dossiers et configuration du fuseau horaire)

In [ ]:
# --- CONFIGURATION ---
DATA_DIR = Path.cwd() / "data"                                                  # définit le dossier "data" dans le répertoire de travail actuel (Current Working Directory)
DATA_DIR.mkdir(exist_ok=True)                                                   # crée le dossier s'il n'existe pas déjà (évite une erreur si le dossier est déjà là)
LOG_FILE = DATA_DIR / "historique_vad.csv"                                      # définit le chemin complet du fichier CSV où seront stockées les valeurs des analyses VAD

In [ ]:
# --- FONCTION D'ENREGISTREMENT DES RESULTATS D'ANALYSE DANS LE CSV ---
def log_vad_data(timestamp: str, v: float, a: float, d: float) -> None:
  """
  Enregistre les valeurs VAD et le timestamp dans un fichier CSV (historique)

  Étapes :
  - Si le fichier n'existe pas
      - Alors crée le fichier avec les en-têtes (headers)
  - Sinon
      - Alors ajoute simplement la nouvelle ligne à la fin (mode append)
  - Empêche l'ajout d'index numériques pour garder un fichier propre

  Args:
      timestamp (str): Date et heure de la mesure
      v (float): Valeur de la Valence
      a (float): Valeur de l'Arousal
      d (float): Valeur de la Dominance

  Returns:
      None: La fonction ne renvoie rien
  """
  df = pd.DataFrame([[timestamp, v, a, d]],                                     # création du DataFrame pour la nouvelle ligne
                    columns=['timestamp', 'valence', 'arousal', 'dominance'])
  df.to_csv(LOG_FILE,
            mode='a',                                                           # ouverture du fichier en mode "append", ajout à la fin sans écraser le contenu existant
            header=not LOG_FILE.exists(),                                       # ajoute l'en-tête uniquement à la création du fichier
            index=False)                                                        # empêche l'écriture des index numériques de pandas

Test analyse flux video en direct sans enregistrement:

In [ ]:
# --- BOUCLE PRINCIPALE : DETECTION, RECONNAISSANCE ET ANALYSE ÉMOTIONNELLE (VAD) ---
def run_face_analysis_loop(facial_recognition_enabled: bool, mean_encoding: np.ndarray) -> None:
  """
  Exécute la boucle temps réel de capture, de filtrage et d'analyse émotionnelle.

  Étapes :
  - Capture automatique d'images via le flux caméra
  - Détection multi-visages optimisée par réduction d'échelle (Resize x0.25)
  - Filtrage par reconnaissance faciale :
      - Mode Anonyme : Accepte uniquement si un seul visage est présent
      - Mode Utilisateur : Compare les signatures pour identifier l'utilisateur autorisé
  - Analyse VAD via Deep Learning (PyTorch) :
      - Normalisation spatiale (crop carré) et centrage du visage
      - Inférence sur le modèle de régression VAD
      - Dénormalisation des scores (Valence, Arousal, Dominance)
  - Journalisation : Sauvegarde des scores dans le CSV et, optionnellement, de l'image du visage

  Args:
      facial_recognition_enabled (bool): Active/Désactive le filtrage par identité
      mean_encoding (np.ndarray): Signature faciale de référence du patient

  Returns:
      None: La boucle s'arrête via une interruption clavier (KeyboardInterrupt)
  """
  # --- BOUCLE DE RECONNAISSANCE ---
  print("\n--- SYSTEME DE CLASSIFICATION ACTIVÉ ---")
  try:
    while True:                                                                 # boucle infinie
      frame = take_photo_auto()                                                 # prise d'une photo via la caméra
      if frame is None: break                                                   # arrête la boucle si la capture échoue
      h, w = frame.shape[:2]
      rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
      # face_locations = face_recognition.face_locations(rgb_frame)             # détecte les coordonnées (haut, droite, bas, gauche) de tous les visages présents
      small_frame = cv2.resize(rgb_frame, (0, 0), fx=0.25, fy=0.25)             # image réduite par 4 (fx=0.25, fy=0.25)
      face_locations_small = face_recognition.face_locations(small_frame)       # la détection se fait sur l'image miniature
      face_locations = [(t*4, r*4, b*4, l*4) for (t, r, b, l) in face_locations_small]  # multiplication des coordonnées par 4 pour retrouver les positions réelles
      num_faces = len(face_locations)
      # clear_output(wait=True)

      # --- CREATION D'UNE COPIE POUR LE DESSIN ---
      display_frame = frame.copy()                                              # conserve 'frame' intacte pour le crop et dessine sur 'display_frame'

      # --- LOGIQUE DE FILTRAGE ET RECONNAISSANCE ---
      best_face_idx = 0
      is_recognized = False
      face_encodings = []
      divisor = max(EPSILON, CONFIDENCE_THRESHOLD)

      if num_faces > 1 and not facial_recognition_enabled:                      # ne traite pas si 2+ visages ET reconnaissance désactivée
        clear_output(wait=True)
        print(f"attente : {num_faces} visages détectés. Mode anonyme limite à 1")
      elif num_faces > 0:
        clear_output(wait=True)
        if facial_recognition_enabled and mean_encoding is not None:
          face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)
          if len(face_encodings) > 0:                                           # recherche du visage appris si reconnaissance activée
            distances = face_recognition.face_distance(np.array(face_encodings), mean_encoding) # s'assure que mean_encoding est bien un array et on compare en calcule l'écart mathématique entre le visage vu et le visage autorisé (0 = même visage)
            best_face_idx = np.argmin(distances)
            score = max(0, min(1, 1 - (distances[best_face_idx] / divisor)))    # normalisation du score, transforme la distance (0 = parfait) en score de confiance (1 = 100% match), on divise par 0.6 (seuil de tolérance) et on utilise max/min pour rester entre 0 et 1
            if score > CONFIDENCE_THRESHOLD:
              is_recognized = True
      else:
        clear_output(wait=True)
        print("aucun visage détecté. En attente ...")

      # --- LOGIQUE DE NOM ET DE COULEUR ---
      for i, (top, right, bottom, left) in enumerate(face_locations):           # parcourt chaque visage
        # Mode Détection Simple (par défaut)
        name = "VISAGE"
        color = (0, 0, 0)                                                       # définit la couleur par défaut du rectangle, ici noire (format BGR)

        # Mode Reconnaissance (Si actif et visage encodé)
        if facial_recognition_enabled and len(face_encodings) > 0:
          face_encoding = face_encodings[i]
          distance = face_recognition.face_distance([mean_encoding], face_encoding)[0] # calcule l'écart mathématique entre le visage vu et le visage autorisé (0 = même visage)
          score = max(0, min(1, 1 - (distance / divisor)))                      # normalisation du score : transforme la distance (0 = parfait) en score de confiance (1 = 100% match), on divise par 0.6 (seuil de tolérance) et on utilise max/min pour rester entre 0 et 1
          if score > CONFIDENCE_THRESHOLD:
            name = f"USER ({score:.2f})"
            color = (100, 150, 0)                                               # définit la couleur du rectangle, ici vert (format BGR) pour un visage autorisé
          else:
            name = f"ALIEN ({score:.2f})"
            color = (0, 0, 200)                                                 # définit la couleur du rectangle, ici rouge si le visage est inconnu ou le score trop bas

        # --- CALCULS ET DESSIN DYNAMIQUES ---
        face_width = right - left                                               # calcule de la largeur du visage
        font_scale = max(0.3, min(face_width / 300, 0.8))                       # calcul de l'échelle de la police avec un sécurité pour éviter que la police soit trop petite ou trop grande
        banner_height = int(face_width / 5)                                     # ajustement de la hauteur du bandeau proportionnellement
        (text_width, text_height), _ = cv2.getTextSize(name, cv2.FONT_HERSHEY_DUPLEX, font_scale, 1) # calcul de la taille du texte pour le centrage
        text_x = left + int((face_width - text_width) / 2)                      # calcule de la position X pour centrer le texte par rapport à la boîte
                                                                                # centre_Boite = left + (face_width / 2)
                                                                                # Position_X_Texte = Centre_Boite - (Largeur_Texte / 2)
        text_y = max(text_height + 5, top - int((banner_height - text_height) / 2)) # calcul de la position Y pour centrer verticalement dans le bandeau mais on s'assure que text_y est au moins à 'text_height' pour ne pas sortir en haut

        # --- DESSIN ---
        cv2.rectangle(display_frame,
                      (left, top),
                      (right, bottom),
                      color,
                      thickness + 1)                                            # cadre: autour du visage détecté avec la couleur définie
        cv2.rectangle(display_frame,
                      (max(0, left - thickness), max(0, top - banner_height)),
                      (min(w, right + thickness), top),
                      color,
                      cv2.FILLED)                                               # bandeau: dessine un bandeau plein au-dessus du cadre pour servir de fond au texte avec la hauteur adaptée
        cv2.putText(display_frame,
                    name,
                    (text_x, text_y),
                    cv2.FONT_HERSHEY_DUPLEX,
                    font_scale,
                    (255, 255, 255),
                    1)                                                          # texte: inscrit le nom (VISAGE ou USER ou ALIEN) et le score de confiance sur l'image avec l'échelle dynamique si la reconnaissance est activée
      cv2_imshow(display_frame)                                                 # affiche l'image finale avec les annotations

      # --- ANALYSE VAD (SANS ENREGISTREMENT) ---
      show_crop = (num_faces == 1 and not facial_recognition_enabled) or (is_recognized and facial_recognition_enabled)

      if show_crop and len(face_locations) > best_face_idx:
        t, r, b, l = face_locations[best_face_idx]                              # coordonnées des 4 angles du visage: top, right, bottom et left

        # --- NORMALISATION SPATIALE ET CENTRAGE DU VISAGE (FORMAT CARRÉ) ---
        side = max(b - t, r - l) + (padding * 2)                                # le plus grand côté est utilisé pour faire un carré
        center_y, center_x = (t + b) // 2, (l + r) // 2                         # calcul des coordonnées du centre du visage
        y1 = max(0, center_y - side // 2)                                       # nouvelles coordonnées des coins du carré basé sur la plus grande dimension pour éviter de déformer l'image au resize
        y2 = min(h, center_y + side // 2)
        x1 = max(0, center_x - side // 2)
        x2 = min(w, center_x + side // 2)

        face_crop = frame[y1:y2, x1:x2]
        face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)                   # conversion pour PyTorch (BGR -> RGB)
        pil_img = Image.fromarray(face_rgb)
        input_tensor = transform(pil_img).unsqueeze(0).to(device)               # le transform() s'occupe de Resize(48,48) + ToTensor + Normalize; .unsqueeze(0) ajoute dimension Batch, fait la même chose que .view(-1, c, h, w)
        with torch.no_grad():                                                   # calcul VAD, fait la même chose que Variable(input_tensor, volatile=True)
          y_norm = model_VAD(input_tensor).squeeze(0)   # shape: [3] -> [V, A, D] normalisés
          # label_std_vad = torch.tensor([1.3228799104690552, 1.2825294733047485, 1.3567899465560913]).to(device)
          # label_mean_vad = torch.tensor([-0.31888335943222046, 0.46993690729141235, 0.09596027433872223]).to(device)
          y = y_norm * label_std_vad + label_mean_vad  # denormalisation vers l'échelle originale

        v, a, d = y.detach().cpu().numpy().tolist()
        v = (v+2)/4                                                             # normalisation entre 0 et 1
        a = (a+2)/4
        d = (d+2)/4

        # --- ENREGISTREMENT DANS LE CSV ---
        current_time = time.strftime('%Y-%m-%d %H:%M:%S')
        log_vad_data(current_time, v, a, d)

        print(f"\n--- ANALYSE EN DIRECT ---")
        face_crop_view = cv2.resize(face_crop, (WIDTH, HEIGHT))
        if NB_MODE: face_crop_view = cv2.cvtColor(face_crop_view, cv2.COLOR_BGR2GRAY)
        debug_view = cv2.resize(face_crop_view, (WIDTH*4, HEIGHT*4), interpolation=cv2.INTER_NEAREST) # affichage zoomé de l'image crop autour du visage sans lisser les pixels pour une bonne visibilité dans Colab
        cv2_imshow(debug_view)
        print(f"[{time.strftime('%H:%M:%S')}] V:{v:.3f} | A:{a:.3f} | D:{d:.3f}")
        msg_save = " | image non enregistrée"
        if SAVE_IMAGE == 1:                                                     # gestion de l'enregistrement de l'image si activé
          filename = f"face_{time.strftime('%Y%m%d_%H%M%S')}.jpg"               # création du nom du fichier unique car avec date et heure
          cv2.imwrite(filename, face_crop_view)                                 # enregistre l'image du visage découpé sur le disque sous le nom défini par 'filename'
          msg_save = f" | image enregistrée : {filename}"
        print(f"[{time.strftime('%H:%M:%S')}] visage traité ({WIDTH}x{HEIGHT} px, Mode NB: {NB_MODE}){msg_save}")

      else:
        print("le programme attend un visage valide")
      time.sleep(DELAY)
  except KeyboardInterrupt:
    print("\nArrêt du programme.")

In [ ]:
# --- MAIN ---
# enabled, signature = setup_recognition()
# run_face_analysis_loop(enabled, signature)
run_face_analysis_loop(*setup_recognition())                                    # lance la fonction d'analyse faciale en décompactant (unpacking) avec * les résultats du setup comme arguments (enabled, signature)

Envoi data VAD par mail dans CSV:

https://www.youtube.com/watch?v=q0oFKa8Skvk
* créer une adresse gmail
* activer 2FA
* générer un "Mot de passe d'application" (App Password) et le copier coller dans le fichier config

In [ ]:
# --- IMPORTATION DES BIBLIOTHEQUES ---
# gestion et analyse de données
import pandas as pd                                                             # pour manipuler et analyser des données structurées comme des fichiers CSV
import numpy as np                                                              # pour les calculs avec les matrices d'images

# gestion du temps et du système
from datetime import datetime, timedelta                                        # pour manipuler des dates et calculer des intervalles de temps
from pathlib import Path                                                        # pour manipuler les chemins des fichiers
import os                                                                       # pour interagir avec le système (gestion des fichiers, dossiers et configuration du fuseau horaire)
import time                                                                     # pour gérer les pauses et les délais dans l'exécution

# envoi par email
import smtplib                                                                  # protocole de connexion au serveur de messagerie
from email.mime.multipart import MIMEMultipart                                  # conteneur pour l'email (objet, corps, pièces jointes)
from email.mime.text import MIMEText                                            # gestion du texte du message (brut ou HTML)
from email.mime.base import MIMEBase                                            # gestion des fichiers joints (images, CSV, etc.)
from email import encoders                                                      # encodage des pièces jointes pour l'envoi

# --- INSTALLATION CONDITIONNELLE ---
# planification de tâches
try:
  import schedule                                                               # pour automatiser l'exécution de fonctions à intervalles réguliers
  print("schedule est déjà installé")
except ImportError:
  print("installation de schedule ...")
  !pip install schedule
  import schedule

In [ ]:
# --- CONFIGURATION ---
DATA_DIR = Path.cwd() / "data"                                                  # répertoire de travail actuel utilisé (Current Working Directory)
DATA_DIR.mkdir(exist_ok=True)
LOG_FILE = DATA_DIR / "historique_vad.csv"
CONFIG_FILE = DATA_DIR / "email_config.txt"

In [ ]:
# --- FONCTION DE GENERATION DE DONNEES FICTIVES (TEST) ---
def generate_fake_vad_data(num_points: int = 24, num_outliers: int = 3, seed: int = 42) -> None:
    """
    Génère des données VAD simulées avec une tendance progressive et des outliers prédéfinis forcés

    Etapes :
    - Nettoie l'historique existant en supprimant le fichier LOG_FILE
    - Utilise une seed pour garantir la reproductibilité des tests
    - Génère une série temporelle où les premiers points suivent une progression linéaire
    - Force l'insertion d'outliers

    Args:
        num_points (int): Nombre total de mesures à générer (par défaut 24)
        num_outliers (int): Nombre de points aberrants à inclure à la fin (par défaut 3)
        seed (int): Valeur d'initialisation du générateur aléatoire (par défaut 42)

    Returns:
        None: La fonction ne renvoie rien
    """
    if LOG_FILE.exists():                                                       # vérifie si un ancien fichier de log existe déjà
        LOG_FILE.unlink()                                                       # supprime l'ancien fichier

    np.random.seed(seed)                                                        # la seed est fixée pour avoir des résultats reproductibles
    start_time = datetime.now() - timedelta(days=1)                             # définit le point de départ de la simulation à 24h dans le passé
    num_normals = num_points - num_outliers

    for i in range(num_points):                                                 # boucle pour créer le nombre de points demandé (par défaut 24)
        timestamp = start_time + timedelta(hours=i)

        if i < num_normals:
            # --- POINTS NORMAUX (qui suivent une tendance) ---
            base = i / num_normals                                              # création d'une corrélation artificielle pour que la régression ait du sens
            v = np.clip(base + np.random.normal(0, 0.15), 0, 1)
            a = np.clip(base + np.random.normal(0, 0.15), 0, 1)
            d = np.clip(base + np.random.normal(0, 0.15), 0, 1)
        else:
            # --- OUTLIERS ---
            if i == num_points - 3:
                v, a, d = 0.9, 0.1, 0.1
            elif i == num_points - 2:
                v, a, d = 0.1, 0.9, 0.9
            else:
                v, a, d = 0.05, 0.95, 0.05

        log_vad_data(timestamp, float(v), float(a), float(d))                   # enregistre ces données simulées dans le fichier CSV
    # print(f"Génération data points pour démo avec succès : {num_points} points de données simulés générés dans {LOG_FILE}")

In [ ]:
# --- FONCTION DE CREATION DU FICHIER DE CONFIGURATION ---
def setup_config_file() -> None:
  """
  Création du fichier de configuration par défaut

  Etapes :
  - S'assure de l'existence du répertoire de données
  - Si le fichier CONFIG_FILE est absent
      - Alors génère un fichier modèle avec des placeholders
  - Sinon, informe l'utilisateur que la configuration existe déjà

  Global Variables:
    - DATA_DIR (Path): Répertoire racine pour le stockage
    - CONFIG_FILE (Path): Chemin complet du fichier .txt de configuration

  Returns:
      None: La fonction ne renvoie rien
  """
  DATA_DIR.mkdir(exist_ok=True)                                                 # si le répertoire de travail n'existe pas, il le crée

  if not CONFIG_FILE.exists():                                                  # vérifie si le fichier de configuration est absent
    print("création du fichier de configuration par défaut ...")

    default_content = """sender_email=ton_email@gmail.com
password=ton_mot_de_passe_application
receiver_emails=medecin_traitant@email.com
cc_emails=copie1@email.com,copie2@email.com
patient_id=23016892500820
"""
    with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
      f.write(default_content)
    print(f"fichier {CONFIG_FILE} créé \nveuillez le modifier avec les vraies valeurs")
  else:
    print(f"le fichier {CONFIG_FILE} existe déjà")

setup_config_file()

In [ ]:
# --- CONFIGURATION DU FUSEAU HORAIRE ---
def decalage_utc() -> None:
  """
    Synchronise l'horloge du système sur le fuseau horaire de Paris

    Etapes :
    - Modifie la variable d'environnement 'TZ' pour cibler 'Europe/Paris'
    - Applique physiquement le changement au niveau du noyau Python avec tzset()
    - Garantit que datetime.now() renvoie l'heure française (gestion été/hiver incluse)

    Returns:
        None: La fonction ne renvoie rien
    """
  os.environ['TZ'] = 'Europe/Paris'                                             # utiliser l'heure de Paris
  time.tzset()                                                                  # applique le changement de fuseau horaire
  now_local = datetime.now()                                                    # récupère l'heure locale (Paris) et l'heure UTC

In [ ]:
# --- PARAMETRES MODIFIABLES ---
HEURE_LOCALE_CIBLE = "11:35:55"
dernier_envoi_date = None
MODE_DEMO = False

In [ ]:
# --- FONCTION D'ENVOI DU RAPPORT JOURNALIER PAR EMAIL ---
def send_once_report() -> None:
  """
  Gère l'envoi unique du rapport quotidien des mesures VAD par email.

  Etapes :
  - Vérifie si un envoi a déjà été effectué aujourd'hui pour éviter les doublons
  - Charge les identifiants et paramètres depuis le fichier de configuration
  - Prépare l'email avec les destinataires et l'objet daté
  - Joint le fichier LOG_FILE (CSV) au message
  - Expédie le mail via le serveur SMTP de Gmail
  - Réinitialise (vide) le fichier CSV après succès et met à jour la date du dernier envoi

  Returns:
      None: La fonction ne renvoie rien
  """
  global dernier_envoi_date
  aujourdhui = datetime.now().date()
  if dernier_envoi_date == aujourdhui:                                          # vérification, si on a déjà envoyé un mail aujourd'hui, on annule
    return

  print(f"[{datetime.now()}] préparation de l'envoi du rapport ...")

  # --- LECTURE DE LA CONFIGURATION ---
  config = {}
  try:
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
      for line in f:
        if "=" in line:
          key, val = line.strip().split("=", 1)                                 # sépare la clé et la valeur au =
          config[key] = val                                                     # stocke les identifiants dans un dictionnaire
  except Exception as e:
    print(f"erreur lors de la lecture du fichier config : {e}")
    return

  # --- VERIFICATION PRESENCE DES DONNEES ---
  if not LOG_FILE.exists():                                                     # vérifie si le fichier CSV existe
    print("erreur fichier de données introuvable")
    return

  # --- CREATION DU MAIL ---
  msg = MIMEMultipart()

  # --- RECUPERATION DES DONNEES DE LA CONFIGURATION ---
  sender = config.get("sender_email")
  receiver = config.get("receiver_emails")
  cc_list = config.get("cc_emails", "")                                         # récupère les adresses mails à mettre en cc s'ils sont là
  patient_num = config.get("patient_id", "Non spécifié")
  password = config.get("password")

  msg['From'] = sender
  msg['To'] = receiver
  if cc_list:
    msg['Cc'] = cc_list

  msg['Subject'] = f"Rapport Émotionnel - Patient {patient_num} - {datetime.now().strftime('%d/%m/%Y')}" # création de l'object du mail

  body = f"Bonjour,\n\nVeuillez trouver ci-joint le relevé des mesures VAD pour le patient n°{patient_num}."
  msg.attach(MIMEText(body, 'plain'))

  """
  msg['From'] = config.get("sender_email")
  msg['To'] = config.get("receiver_emails")
  msg['Subject'] = f"Rapport Émotionnel - {datetime.now().strftime('%d/%m/%Y')}"

  body = "Bonjour,\n\nVeuillez trouver ci-joint le relevé des mesures VAD."
  msg.attach(MIMEText(body, 'plain'))
  """

  # --- AJOUT DU FICHIER CSV EN PIECE JOINTE ---
  try:
    with open(LOG_FILE, "rb") as attachment:
      part = MIMEBase("application", "octet-stream")
      part.set_payload(attachment.read())
      encoders.encode_base64(part)
      part.add_header("Content-Disposition", f"attachment; filename={LOG_FILE.name}")
      msg.attach(part)
  except Exception as e:
    print(f"erreur lors de l'ajout de la pièce jointe : {e}")
    return

  # --- ENVOI SMTP ---
  try:
    with smtplib.SMTP("smtp.gmail.com", 587) as server:
      server.starttls()
      server.login(sender, password)
      destinataires = [receiver]
      if cc_list and isinstance(cc_list, str):                                  # vérification que cc_list n'est pas None et n'est pas une chaîne vide
        destinataires.extend([email.strip() for email in cc_list.split(",")])
      server.sendmail(sender, destinataires, msg.as_string())
      print("rapport journalier envoyé avec succès")
      # --- VIDER LE CSV APRES ENVOI REUSSI ---
      try:
        with open(LOG_FILE, 'w', encoding='utf-8') as f:
          f.write("timestamp,valence,arousal,dominance\n")                      # réinitialise le fichier avec seulement l'en-tête
        print("fichier CSV réinitialisé avec succès")
      except Exception as e_clean:
        print(f"erreur lors du vidage du fichier CSV : {e_clean}")
      # validation de la date d'envoi même si le vidage a échoué pour éviter de renvoyer le mail en boucle
      dernier_envoi_date = aujourdhui                                           # la date du dernier envoi réussi est update
  except Exception as e:
    print(f"erreur lors de l'envoi SMTP : {e}")

In [ ]:
# --- CONFIGURATION DU PLANIFICATEUR ---
def setup_scheduler() -> None:
  """
  Initialise la tâche récurrente d'envoi du rapport médical

  Etapes :
  - Enregistre la fonction 'send_once_report' dans la file d'attente de la bibliothèque schedule
  - Configure la fréquence (quotidienne) et l'heure précise (basée sur HEURE_LOCALE_CIBLE)
  - Affiche un état des lieux dans la console pour vérifier la synchronisation horaire

  Global Variables:
    - HEURE_LOCALE_CIBLE (str)
    - send_once_report (function): La fonction qui génère et envoie l'email

  Returns:
      None: La fonction ne renvoie rien
  """
  schedule.every().day.at(HEURE_LOCALE_CIBLE).do(send_once_report)              # planification de l'envoi du rapport journalier
  print("planificateur démarré")
  print(f"heure actuelle du serveur (corrigée sur Paris): {datetime.now().strftime('%H:%M:%S')}")
  print(f"envoi programmé pour : {HEURE_LOCALE_CIBLE} (heure locale)")

In [ ]:
# --- FONCTION DE TEST DEMONSTRATION --
def run_demo() -> None:
  """
  Exécute un cycle complet du système (Simulation de mesure -> Envoi) pour démonstration

  Étapes :
  - Force la réinitialisation du verrou 'dernier_envoi_date' pour contourner la limite d'un mail par jour
  - Génère un jeu de données fictives complet (24 points) dans le fichier LOG_FILE
  - Déclenche immédiatement l'envoi de l'email
  - Réinitialise à nouveau le verrou après l'envoi pour ne pas bloquer le planificateur (scheduler)

  Global Variables:
    - dernier_envoi_date (datetime.date): Variable de contrôle anti-spam journalier

  Returns:
      None: La fonction ne renvoie rien
  """
  global dernier_envoi_date
  print("\n--- MODE DEMO ACTIVÉ ---")
  dernier_envoi_date = None                                                     # réinitialisation du verrou pour autoriser envoi du mail même si déjà fait aujourd'hui
  generate_fake_vad_data(num_points=24)                                         # remplir le CSV de données aléatoires
  send_once_report()                                                            # envoi instantané
  dernier_envoi_date = None                                                     # réinitialisation du verrou pour ne pas empecher de renvoyer le mail dans la boucle auto
  print("--- FIN DU TEST DEMO ---\n")

In [ ]:
# --- MAIN ---
decalage_utc()                                                                  # mise à jour du fuseau horaire
if MODE_DEMO:
  run_demo()
else:
  setup_scheduler()                                                             # initialisation du planificateur
  while True:
    schedule.run_pending()                                                      # vérifie si il y a des tâches à exécuter
    time.sleep(30)                                                              # vérifie toutes les 30 secondes si c'est l'heure

Test threadding

In [ ]:
# --- IMPORTATION DES BIBLIOTHEQUES ---
"""
import threading                                                                # pour la gestion des threads et des tâches parallèles
import time                                                                     # pour la gestion du temps
"""

In [ ]:
"""
def tache_a():
  print("tâche A démarrage")
  for i in range(5):
    time.sleep(1)
    print(f"tâche A étape {i+1}/5")
  print("tâche A terminée")

def tache_b():
  print("tâche B démarrage")
  for i in range(5):
    time.sleep(0.7)                                                             # plus rapide que tâche A
    print(f"tâche B étape {i+1}/5")
  print("tâche B terminée!")

thread1 = threading.Thread(target=tache_a)                                      # création des threads
thread2 = threading.Thread(target=tache_b)
"""

In [ ]:
# --- MAIN ---
"""
print("début du test global")
thread1.start()                                                                 # démarrage des threads
thread2.start()
thread1.join()
thread2.join()
print("fin du test global")
"""

Graph interractif 3D:

In [ ]:
# --- IMPORTATION DES BIBLIOTHEQUES ---
import numpy as np                                                              # pour les calculs avec les matrices
import pandas as pd                                                             # pour la structuration + analyse de données
import plotly.graph_objects as go                                               # pour la création de graphiques 3D interactifs
from IPython.display import display, Javascript, clear_output                   # pour l'affichage des graphiques dans le notebook

In [ ]:
# --- FONCTION DE CHARGEMENT DES DONNEES POUR VISUALISATION ---
def load_vad_data() -> tuple[np.ndarray, list]:
  """
  Charge et formate les données du fichier CSV pour la visualisation graphique.

  Etapes :
  - Vérifie l'existence physique du fichier de données (LOG_FILE)
  - Importe les mesures VAD (valence, arousal, dominance) du fichier CSV via pandas
  - Extrait les coordonnées VAD (Valence, Arousal, Dominance) en format NumPy
  - Convertit les horodatages en objets temporels pour un formatage lisible
  - Génère une liste d'étiquettes personnalisées pour identifier chaque point

  Returns:
      tuple: (points_max, names_max)
          - points_max (np.ndarray): Matrice des scores VAD
          - names_max (list): Liste des labels pour chaque point
  """
  if not LOG_FILE.exists():                                                     # vérifie si le fichier de log est absent avant de tenter la lecture
    print(f"erreur, le fichier {LOG_FILE} n'existe pas")
    return np.array([]), []                                                     # retourne des listes vides pour éviter de faire planter le code suivant
  df = pd.read_csv(LOG_FILE)                                                    # importe les données du fichier CSV dans un tableau pandas
  points_max = df[['valence', 'arousal', 'dominance']].to_numpy()               # préparation de points_max (V, A, D) en sélectionnant uniquement les colonnes de données, conversion en array NumPy
  df['timestamp'] = pd.to_datetime(df['timestamp'])                             # convertit les dates du format texte en objets temporels Python
  names_max = [f"P_{i+1} ({t.strftime('%d/%m/%Y %H:%M')})" for i, t in enumerate(df['timestamp'])] # création des labels pour chaque point de données
  return points_max, names_max

# print (load_vad_data())

In [ ]:
# --- FONCTION DE CALCUL DE LA LIGNE DE TENDANCE ---
def get_regression_line(pts: np.ndarray) -> tuple:
    """
    Calcule la ligne de tendance (régression linéaire 3D) par décomposition en valeurs singulières (SVD)

    Etapes :
    - Vérifie que le nombre de points est suffisant (minimum 2) pour définir une droite
    - Calcule le centroïde (moyenne) du nuage de points pour centrer les données
    - Applique une SVD pour identifier le vecteur propre associé à la plus grande valeur singulière
    - Projette les points sur cet axe pour déterminer l'étendue de la ligne (t_min à t_max)
    - Génère un ensemble de points interpolés pour le tracé graphique 3D

    Args:
        pts (np.ndarray): Matrice des points de données (N points x 3 dimensions)

    Returns:
        tuple: (line_points, centroid, direction, vh)
            - line_points (np.ndarray): Coordonnées des points formant la ligne de tendance
            - centroid (np.ndarray): Centre de gravité du nuage de points
            - direction (np.ndarray): Vecteur directeur de la ligne (composante principale)
            - vh (np.ndarray): Matrice complète des vecteurs propres
    """
    if len(pts) < 2:                                                            # vérifie qu'il y a assez de points pour tracer une ligne
        return np.array([[None, None, None]]), np.array([None]), None, None
    centroid = pts.mean(axis=0)                                                 # calcul du centre de gravité des points
    u, s, vh = np.linalg.svd(pts - centroid)                                    # décomposition en valeurs singulières pour trouver l'axe principal
    direction = vh[0, :]                                                        # extraction du vecteur directeur de la ligne
    projections = np.dot(pts - centroid, direction)                             # création de points le long de cet axe pour le tracé
    t_min = projections.min()                                                   # projection des points sur la ligne pour trouver les t_min et t_max
    t_max = projections.max()
    line_points = centroid + np.outer(np.linspace(t_min - 0.1, t_max + 0.1, 10), direction) # la ligne s'étend un peu plus loin que les points extrêmes pour une meilleure visualisation
    return line_points, centroid, direction, vh                                 # retourne aussi le centroïde et la direction pour le calcul des outliers

In [ ]:
def detect_outliers(points: np.ndarray, centroid: np.ndarray, direction: np.ndarray, threshold_multiplier: float = 2.0) -> tuple:
    """
    Identifie les points dont l'écart à la ligne de tendance est statistiquement anormal.

    Étapes :
    - Calcule la distance orthogonale de chaque point à la droite de régression via le produit vectoriel
    - Détermine le seuil d'exclusion basé sur la moyenne et l'écart-type des distances
    - Filtre les points en deux groupes : 'inliers' (normaux) et 'outliers' (atypiques)
    - Associe les noms/horodatages aux points identifiés comme outliers

    Args:
        points (np.ndarray): Le nuage de points VAD actuel
        centroid (np.ndarray): Le centre de gravité calculé par la régression
        direction (np.ndarray): Le vecteur directeur de la ligne de tendance
        threshold_multiplier (float): Sensibilité de détection (par défaut 2.0 sigma)

    Returns:
        tuple: (inliers, outliers, outlier_names, distances, outlier_threshold)
            - inliers/outliers (np.ndarray): Points segmentés
            - outlier_names (np.array): Labels des points atypiques
            - distances (np.ndarray): Liste de toutes les distances calculées
            - outlier_threshold (float): La valeur de distance limite calculée
    """
    if len(points) < 2:
        return np.array([]), np.array([]), np.array([])
    # vecteur de chaque point au centroïde
    vec_to_centroid = points - centroid
    # produit vectoriel entre le vecteur (point - centroïde) et le vecteur directeur de la ligne || (Q - P0) x v ||
    cross_product = np.cross(vec_to_centroid, direction)
    # norme du produit vectoriel || (Q - P0) x v ||
    magnitude_cross_product = np.linalg.norm(cross_product, axis=1)
    # norme du vecteur directeur de la ligne || v ||
    magnitude_direction = np.linalg.norm(direction)
    # distances orthogonales de chaque point à la ligne
    distances = magnitude_cross_product / magnitude_direction
    # détection des outliers basée sur l'écart-type des distances
    mean_distance = np.mean(distances)
    std_distance = np.std(distances)
    # un point est un outlier si sa distance est supérieure à (moyenne + k * écart-type)
    outlier_threshold = mean_distance + threshold_multiplier * std_distance
    is_outlier = distances > outlier_threshold
    inliers = points[~is_outlier]
    outliers = points[is_outlier]
    outlier_names = np.array(names_max)[is_outlier]                             # noms des points outliers
    return inliers, outliers, outlier_names, distances, outlier_threshold

In [ ]:
# --- FONCTION DE VISUALISATION DE L'ESPACE EMOTIONNEL (PLOTLY) ---
def plot_descartes_space(points_max, names_max, get_regression_line):

    num_max = len(points_max)

    # --- EXTRACTION DES DATES ---
    start_time = names_max[0].split('(')[-1].replace(')', '')
    end_time = names_max[-1].split('(')[-1].replace(')', '')

    # --- LECTURE DE LA CONFIGURATION ---
    config = {}
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
      for line in f:
        if "=" in line:
          key, val = line.strip().split("=", 1)                                 # sépare la clé et la valeur au =
          config[key] = val                                                     # stocke les identifiants dans un dictionnaire
    patient_num = config.get("patient_id", "Non spécifié")

    # --- POINTS DE REFERENCE (DESCARTES) ---
    significant_emotions = {                                                    # dictionnaire des passions selon le modèle de Descartes
        "Love": [1,1,0], "Hate": [0,1,0], "Admiration": [0.5,1,0.5], "Joy": [1,1,0.5], "Sadness": [0,0,0.5], "Desire": [1,0.5,0.5]
    }
    sig_pts = np.array(list(significant_emotions.values()))                     # conversion des coordonnées des émotions en tableau Numpy

    # --- CREATION DE LA FIGURE ---
    fig = go.Figure()

    # --- 1. TRACE DU PLAN CARRÉ DE RÉFÉRENCE (z=0) ---
    square_x = [0, 1, 1, 0, 0]
    square_y = [0, 0, 1, 1, 0]
    square_z = [0, 0, 0, 0, 0]                                                  # On dessine juste le contour du carré Valence/Arousal
    fig.add_trace(go.Scatter3d(
        x=square_x, y=square_y, z=square_z,
        mode='lines', line=dict(color='gray', width=2),
        name='Plan V-A', showlegend=False, hoverinfo='skip'
    ))

    # --- FONCTION POUR GENERER LES LOLLIPOPS (Tiges) ---
    def add_lollipops(points, colors, group_name="autre", opacity=0.8):
        lx, ly, lz = [], [], []                                                 # On crée des listes de coordonnées entrecoupées de None pour tracer des lignes séparées
        for p in points:
            lx.extend([p[0], p[0], None])
            ly.extend([p[1], p[1], None])
            lz.extend([0, p[2], None])                                          # Part du sol (0) vers la Dominance (z)

        return go.Scatter3d(
            x=lx, y=ly, z=lz,
            mode='lines',
            line=dict(color=colors, width=2),
            hoverinfo='skip',
            hoverlabel=None,
            showlegend=False,
            legendgroup=group_name,
            opacity=opacity
        )

    # --- 2. TRACE DES PASSIONS DE DESCARTES (Points seuls) ---
    fig.add_trace(go.Scatter3d(                                                 # ajout des points (têtes) de référence "Descartes" sur le graphique 3D
        x=sig_pts[:,0], y=sig_pts[:,1], z=sig_pts[:,2],
        mode='markers+text', name='Descartes Passions',
        marker=dict(size=6, color='purple'),
        text=list(significant_emotions.keys()),                                 # affichage des noms des émotions
        textposition="top center",
        hovertemplate="<b>%{text}</b><br>Valence: %{x:.2f}<br>Arousal: %{y:.2f}<br>Dominance: %{z:.2f}<extra></extra>"
    ))

    # calcul des inliers et outliers
    initial_line_points, initial_centroid, initial_direction, _ = get_regression_line(points_max)
    initial_inliers, initial_outliers, initial_outlier_names, initial_distances, initial_outlier_threshold = detect_outliers(points_max, initial_centroid, initial_direction)

    # --- 3. TRACE DES POINTS DYNAMIQUES (Points + Tiges) ---
    # Trace 3.1: Points inliers
    fig.add_trace(add_lollipops(initial_inliers, 'green', "inliners", 0.6))     # Tiges
    fig.add_trace(go.Scatter3d(                                                 # Têtes
        x=initial_inliers[:,0], y=initial_inliers[:,1], z=initial_inliers[:,2],
        mode='markers', name='Points Inliers',
        legendgroup="inliners",
        marker=dict(size=5, color='green', opacity=0.8),
        text=[names_max[i] for i, is_outlier in enumerate(initial_distances > initial_outlier_threshold) if not is_outlier],
        hovertemplate="<b>%{text}</b><br>Valence: %{x:.2f}<br>Arousal: %{y:.2f}<br>Dominance: %{z:.2f}<extra></extra>"
    ))

    # Trace 3.2: Points outliers
    fig.add_trace(add_lollipops(initial_outliers, 'red', "outliers", 0.6))      # Tiges
    fig.add_trace(go.Scatter3d(                                                 # Têtes
        x=initial_outliers[:,0], y=initial_outliers[:,1], z=initial_outliers[:,2],
        mode='markers', name='Points Outliers',
        legendgroup="outliers",
        marker=dict(size=8, color='red', symbol='x', opacity=0.8),              # points rouges en forme de 'x' pour les outliers
        text=initial_outlier_names,
        hovertemplate="<b>%{text}</b><br>Valence: %{x:.2f}<br>Arousal: %{y:.2f}<br>Dominance: %{z:.2f}<extra></extra>"
    ))

    # --- 4. TRACE LIGNE DE TENDANCE ---
    initial_line, _, _, _ = get_regression_line(initial_inliers)
    fig.add_trace(go.Scatter3d(
        x=initial_line[:,0], y=initial_line[:,1], z=initial_line[:,2],
        mode='lines', name='Régression (inliers seulement)',
        line=dict(color='blue', width=4),
        hoverinfo='skip'                                                        # désactivation du survol sur la ligne
    ))

    # --- 5. TRACE DROITES DE REFERENCE ---
    fig.add_trace(go.Scatter3d(
        x=[0, 1], y=[0, 1], z=[0, 1],
        mode='lines', name='BD',
        line=dict(color='yellow', width=6),
        hoverinfo='skip'
    ))

    fig.add_trace(go.Scatter3d(
        x=[0, 1], y=[0, 0], z=[0, 0],
        mode='lines', name='MD',
        line=dict(color='deeppink', width=6),
        hoverinfo='skip'
    ))

    # --- 6. CONFIGURATION DU LAYOUT ---
    fig.update_layout(
        title={'text': f"Trajectoire emotionelle (VAD) du patient {patient_num} <br><sup>Du {start_time} au {end_time}</sup>", 'y': 0.95, 'x': 0.5},
        scene=dict(
            # Face X (latérale) -> Transparente
            xaxis=dict(
                range=[-0.02,1.02],
                title="Valence",
                showspikes=False,
                showbackground=False
            ),
            # Face Y (latérale) -> Transparente
            yaxis=dict(
                range=[-0.02,1.02],
                title="Arousal",
                showspikes=False,
                showbackground=False
            ),
            # Face Z (sol) -> Pas Transparente
            zaxis=dict(
                range=[-0.02,1.02], #[0,1],
                title="Dominance",
                showbackground=True,
                zeroline=True,
                showspikes=False,
                showgrid=True,
                backgroundcolor="rgba(0, 0, 255, 0.1)"
            ),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),                         # position initiale de la caméra (zoom arrière pour avoir tout le cube visible)
            aspectmode='cube'                                                   # garantit que le graphique est un cube parfait et le reste
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        height=800,
        updatemenus=[{
            "type": "buttons",
            "showactive": False,                                                # Désactive l'état "enfoncé" visuel
            "active": -1,                                                       # Aucun bouton sélectionné par défaut
            "buttons": [
                {"label": "Stop", "args": [{"scene.camera.eye": None}], "method": "skip"},
                {"label": "Play", "args": [{"scene.camera.eye": None}], "method": "skip"},
                {"label": "Reset View", "args": [{"scene.camera.eye": None}], "method": "skip"}
            ],
            "direction": "left", "pad": {"r": 10, "t": 87}, "x": 0.1, "y": 0
        }]
    )
    return fig

In [ ]:
def enable_plotly_rotation(speed: float = 0.007) -> None:
    """
    Injecte un script JavaScript pour activer la rotation automatique du graphique 3D

    Étapes :
    - Identifie le dernier graphique Plotly rendu dans la cellule du notebook
    - Calcule les paramètres polaires (angle, rayon) à partir de la position caméra actuelle
    - Utilise requestAnimationFrame pour une rotation fluide autour de l'axe Z
    - Implémente des écouteurs d'événements (Play, Stop, Reset View) pour contrôler l'animation
    - Désactive la rotation si l'utilisateur interagit manuellement avec le graphique

    Args:
        speed (float): Vitesse de rotation angulaire. Par défaut 0.007.

    Returns:
        None: La fonction ne renvoie rien
    """
    js_code = f'''
    (function() {{
        var plots = document.querySelectorAll('.js-plotly-plot');
        var gd = plots[plots.length - 1];
        if (!gd) return;

        window.isRotationActive = true;
        var speed = {speed};                                                    // Vitesse de rotation

        // Coordonnées initiales définies dans votre layout Python
        const initialCamera = {{x: -1.5, y: -1.5, z: 1.2}};

        // Variables d'état pour la caméra
        var angle, radius, currentZ;

        function updateCameraParams() {{
            // Récupère la position actuelle (réelle) de la caméra
            var eye = gd.layout.scene.camera.eye;
            currentZ = eye.z;
            radius = Math.sqrt(eye.x * eye.x + eye.y * eye.y);
            angle = Math.atan2(eye.y, eye.x);
        }}

        function rotate() {{
            if (!window.isRotationActive) return;

            angle += speed;
            var x = radius * Math.cos(angle);
            var y = radius * Math.sin(angle);

            // On ne met à jour que X et Y pour tourner autour de l'axe Z
            Plotly.relayout(gd, {{
                'scene.camera.eye': {{x: x, y: y, z: currentZ}}
            }}).then(() => {{
                if (window.isRotationActive) requestAnimationFrame(rotate);
            }});
        }}

        // Arrêt si l'utilisateur manipule le graphe manuellement
        gd.addEventListener('mousedown', function() {{
            window.isRotationActive = false;
        }}, true);

        gd.on('plotly_buttonclicked', function(data) {{
            var label = data.button.label;
            if (label.includes("Play")) {{
                if (!window.isRotationActive) {{
                    updateCameraParams();                                       // On recalcule tout avant de repartir
                    window.isRotationActive = true;
                    rotate();
                }}
            }} else if (label.includes("Stop")) {{
                window.isRotationActive = false;
            }} else if (label.includes("Reset View")) {{
            Plotly.relayout(gd, {{
                'scene.camera.eye': initialCamera
            }}).then(() => {{
                updateCameraParams(); // Recalcule les paramètres sur la nouvelle position
            }});
            }}
        }});
        // Initialisation et lancement automatique
        updateCameraParams();
        rotate();
    }})();
    '''
    display(Javascript(js_code))

In [ ]:
# --- MAIN ---
#generate_fake_vad_data()
points_max, names_max = load_vad_data()

# Affichage du graphique
plot_descartes_space(points_max, names_max, get_regression_line).show(renderer="colab", config={'displayModeBar': True})

# Update du graphique
enable_plotly_rotation(speed=0.01)

2 Dashboards pour médecins et patients

In [ ]:
# --- IMPORTATION DES BIBLIOTHEQUES ---
import shutil

# --- INSTALLATION CONDITIONNELLE ---
try:
  import streamlit
  print("streamlit est déjà installé")
except ImportError:
  print("installation de streamlit...")
  !pip install -q streamlit
  import streamlit

if shutil.which("lt"):
  print("Localtunnel est déjà installé")
else:
  print("installation de Localtunnel...")
  !npm install -q -g localtunnel

  if shutil.which("lt"):
    print("Localtunnel est bien installé")
  else:
    print("Localtunnel ne s'est pas bien installé")

Pour patient

In [ ]:
%%writefile app.py
import streamlit as st
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from textwrap import dedent
from pathlib import Path
from datetime import datetime, timedelta

LOG_FILE = Path("patient_data.csv")
st.set_page_config(page_title="Clinical VAD analysis", layout="wide", page_icon="🧠")

def get_regression_params(points):
    centroid = np.mean(points, axis=0)
    uu, dd, vv = np.linalg.svd(points - centroid)
    direction = vv[0]
    return centroid, direction

def get_line_visual_points(centroid, direction):
    t = np.linspace(-0.5, 0.5, 20)
    return centroid + t[:, np.newaxis] * direction

def detect_outliers(points, centroid, direction, threshold=0.20):
    vectors = points - centroid
    projections = np.outer(np.dot(vectors, direction), direction)
    distances = np.linalg.norm(vectors - projections, axis=1)
    return distances > threshold

# --- GENERATEUR DE DONNEES ---
def generate_fake_vad_data():
    dates = [datetime.now() - timedelta(hours=i) for i in range(30, 0, -1)]
    v = np.linspace(0.2, 0.7, 30) + np.random.normal(0, 0.08, 30)
    a = np.linspace(0.1, 0.6, 30) + np.random.normal(0, 0.08, 30)
    d = np.linspace(0.3, 0.5, 30) + np.random.normal(0, 0.08, 30)
    df_new = pd.DataFrame({'Date': dates, 'Valence': np.clip(v, 0, 1), 'Arousal': np.clip(a, 0, 1), 'Dominance': np.clip(d, 0, 1)})
    df_new.to_csv(LOG_FILE, index=False)

if not LOG_FILE.exists():
    generate_fake_vad_data()

df = pd.read_csv(LOG_FILE)
points = df[['Valence', 'Arousal', 'Dominance']].values
current_pos = df.iloc[-1]

# Score global pour la jauge (Moyenne VAD)
current_score = (current_pos['Valence'] + current_pos['Arousal'] + current_pos['Dominance']) / 3

# Logique de seuils
if current_score > 0.65:
    status_text, status_color, advice = "Phase de haute énergie", "#FF9800", "⚠️ Intensité élevée. Pensez à faire des pauses et des exercices de relaxation."
elif current_score < 0.35:
    status_text, status_color, advice = "Phase de basse énergie", "#1976D2", "🌧️ Repli détecté. Moment difficile, soyez bienveillant envers vous même."
else:
    status_text, status_color, advice = "Phase d'équilibre", "#7CB342", "✅ Zone de stabilité émotionnelle."

# --- INTERFACE : HEADER ET JAUGE ---
st.title("🩺 Your mood evolution")

col_kpi, col_gauge = st.columns([2, 1])

with col_kpi:
    st.markdown(f"## {status_text}")
    st.info(advice)
    if st.button("🔄 Regenerate test data"):
        generate_fake_vad_data()
        st.rerun()

with col_gauge:
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=current_score,
        title={'text': "Global score"},
        gauge={
            'axis': {'range': [0, 1]},
            'bar': {'color': status_color},
            'steps': [
                {'range': [0, 0.35], 'color': "#E3F2FD"}, # Bleu
                {'range': [0.35, 0.65], 'color': "#F1F8E9"}, # Vert
                {'range': [0.65, 1], 'color': "#FFF3E0"}  # Orange
            ]
        }
    ))
    fig_gauge.update_layout(height=250, margin=dict(l=20, r=20, t=50, b=20))
    st.plotly_chart(fig_gauge, use_container_width=True)

st.markdown("---")

# --- VISUALISATION 3D ---
st.subheader("Analysis in the VAD space")

# Calculs de trajectoire
centroid, direction = get_regression_params(points)
is_outlier = detect_outliers(points, centroid, direction)
line_pts = get_line_visual_points(centroid, direction)

fig = go.Figure()

# 1. Lollipops liées (Têtes + Tiges)
def add_linked_lollipops(pts, dates, mask, color, name, legend_id, symbol='circle'):
    filtered_pts = pts[mask]
    filtered_dates = dates[mask]
    lx, ly, lz = [], [], []
    for p in filtered_pts:
        lx.extend([p[0], p[0], None])
        ly.extend([p[1], p[1], None])
        lz.extend([0, p[2], None])

    fig.add_trace(go.Scatter3d(
        x=lx, y=ly, z=lz, mode='lines',
        line=dict(color=color, width=3),
        showlegend=False, hoverinfo='skip', opacity=0.8, legendgroup=legend_id,
    ))
    fig.add_trace(go.Scatter3d(
        x=filtered_pts[:,0], y=filtered_pts[:,1], z=filtered_pts[:,2],
        mode='markers', name=name, legendgroup=legend_id, customdata=filtered_dates,
        marker=dict(size=8, color=color, symbol=symbol),
        hovertemplate="<b>Date: %{customdata}</b><br> <br>Valence: %{x:.2f}<br>Arousal: %{y:.2f}<br>Dominance: %{z:.2f}<extra></extra>"
    ))

dates_series = pd.to_datetime(df['Date']).dt.strftime('%d/%m/%Y %H:%M')
add_linked_lollipops(points, dates_series, ~is_outlier, 'green', 'inliners', 'Reliable Measurements')
add_linked_lollipops(points, dates_series, is_outlier, 'red', 'outliers','Excluded Measurements', symbol='x')

# 2. Axes Cliniques (BD et Dépression)
fig.add_trace(go.Scatter3d(
    x=[0, 1], y=[0, 1], z=[0, 1],
    mode='lines', name='Bipolar Axis',
    line=dict(color='yellow', width=6),
    hoverinfo='skip'
))

fig.add_trace(go.Scatter3d(
    x=[0, 1], y=[0, 0], z=[0, 0],
    mode='lines', name='Depression Axis',
    line=dict(color='deeppink', width=6),
    hoverinfo='skip'
))

# 3. Ligne de tendance
fig.add_trace(go.Scatter3d(
    x=line_pts[:,0], y=line_pts[:,1], z=line_pts[:,2],
    mode='lines', name='Your overall evolution axis',
    line=dict(color='cyan', width=6),
    hoverinfo='skip'
))

# 4. Passions de Descartes (Repères)
descartes = {                                                                   # dictionnaire des passions selon le modèle de Descartes
        "Love": [1,1,0], "Hate": [0,1,0], "Admiration": [0.5,1,0.5], "Joy": [1,1,0.5], "Sadness": [0,0,0.5], "Desire": [1,0.5,0.5]
    }
for label, p in descartes.items():
    fig.add_trace(go.Scatter3d(
        x=[p[0]], y=[p[1]], z=[p[2]],
        mode='markers+text', name='Descartes Passions',
        marker=dict(size=8, color='purple'),showlegend=False,
        text=[label],                                 # affichage des noms des émotions
        textposition="top center",
        hovertemplate="<b>%{text}</b><br>Valence: %{x:.2f}<br>Arousal: %{y:.2f}<br>Dominance: %{z:.2f}<extra></extra>"
    ))

fig.update_layout(
    height=800,
    scene=dict(
      # Face X (latérale) -> Transparente
      xaxis=dict(
          range=[-0.02,1.02],
          title="Valence",
          showspikes=False,
          showbackground=False,
      ),
      # Face Y (latérale) -> Transparente
      yaxis=dict(
          range=[-0.02,1.02],
          title="Arousal",
          showspikes=False,
          showbackground=False,
      ),
      # Face Z (sol) -> Pas Transparente
      zaxis=dict(
          range=[-0.02,1.02], #[0,1],
          title="Dominance",
          showbackground=True,
          zeroline=True,
          showspikes=False,
          showgrid=True,
          backgroundcolor="rgba(83, 83, 255, 0.1)"
      ),
      camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2)),                              # position initiale de la caméra (zoom arrière pour avoir tout le cube visible)
      aspectmode='cube'                                                         # garantit que le graphique est un cube parfait et le reste
  ),
  margin=dict(l=0, r=0, b=0, t=50)
)

st.plotly_chart(fig, use_container_width=True)

In [ ]:
# Lancement de Streamlit et du tunnel
# !streamlit run app.py & npx localtunnel --port 8501
!streamlit run app.py \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    --browser.gatherUsageStats false \
    > /dev/null 2>&1 & npx localtunnel --port 8501 | grep "your url is"

Pour médecin

In [ ]:
%%writefile app.py
import streamlit as st
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
import pandas as pd
import numpy as np
from textwrap import dedent
from pathlib import Path
from datetime import datetime, timedelta

LOG_FILE = Path('patient_data.csv')

def get_regression_params(points):
    centroid = np.mean(points, axis=0)
    uu, dd, vv = np.linalg.svd(points - centroid)
    direction = vv[0]
    return centroid, direction

def get_line_visual_points(centroid, direction):
    t = np.linspace(-0.5, 0.5, 20)
    return centroid + t[:, np.newaxis] * direction

def detect_outliers(points, centroid, direction, threshold=0.20):
    vectors = points - centroid
    projections = np.outer(np.dot(vectors, direction), direction)
    distances = np.linalg.norm(vectors - projections, axis=1)
    return distances > threshold

# --- GENERATEUR DE DONNEES ---
def generate_fake_vad_data(patient_id="PA-001"):
    dates = [datetime.now() - timedelta(hours=i) for i in range(30, 0, -1)]
    v = np.linspace(0.2, 0.7, 30) + np.random.normal(0, 0.08, 30)
    a = np.linspace(0.1, 0.6, 30) + np.random.normal(0, 0.08, 30)
    d = np.linspace(0.3, 0.5, 30) + np.random.normal(0, 0.08, 30)
    df_new = pd.DataFrame({'Date': dates, 'Patient_ID': patient_id, 'Valence': np.clip(v, 0, 1), 'Arousal': np.clip(a, 0, 1), 'Dominance': np.clip(d, 0, 1)})
    df_new.to_csv(LOG_FILE, index=False)

if not LOG_FILE.exists():
    generate_fake_vad_data()

st.set_page_config(page_title="Clinical analysis", layout="wide", page_icon="🧠")

# --- STYLE CSS ---
st.markdown("""
    <style>
    body { background: linear-gradient(135deg, #E3F2FD 0%, #BBDEFB 100%); font-family: 'Segoe UI', sans-serif; }
    .main .block-container { background-color: rgba(255, 255, 255, 0.5); border-radius: 20px; padding: 2rem; }
    .patient-banner {background-color: #0D47A1; color: white; padding: 15px; border-radius: 12px; text-align: center; font-size: 24px; font-weight: bold; margin-bottom: 25px; box-shadow: 0 4px 15px rgba(0,0,0,0.1);}
    h1, h2, h3 { color: #0D47A1; font-weight: 700; }
    .legend-box { background: rgba(255, 255, 255, 0.9); padding: 15px; border-radius: 10px; border: 1px solid #B3D4FC; font-size: 14px; }
    .kpi-card { background: rgba(255, 255, 255, 0.9); border-radius: 14px; padding: 0.9rem 1rem; box-shadow: 0 8px 20px rgba(13, 71, 161, 0.08); }
    .axis-def { background: rgba(255, 255, 255, 0.8); padding: 15px; border-radius: 10px; margin-top: 10px; font-size: 13px; color: #444; border-left: 4px solid #0D47A1; }
    .info-box { background: rgba(255, 255, 255, 0.9); padding: 15px; border-radius: 10px; font-size: 14px; }
    </style>
""", unsafe_allow_html=True)

df = pd.read_csv(LOG_FILE)
df['Date'] = pd.to_datetime(df['Date'])
points = df[['Valence', 'Arousal', 'Dominance']].to_numpy()
patient_id_file = df['Patient_ID'].iloc[0] if 'Patient_ID' in df.columns else "Non défini"

# --- AFFICHAGE DE L'ID RÉCUPÉRÉ TOUT EN HAUT ---
st.markdown(f'<div class="patient-banner">Sujet d\'étude : {patient_id_file}</div>', unsafe_allow_html=True)

# --- SIDEBAR ---
with st.sidebar:
    st.title("Parameters")
    if st.button("🔄 Regenerate test data"):
        generate_fake_vad_data()
        st.rerun()

corr = df[['Valence', 'Arousal', 'Dominance']].corr()
avg_corr = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().mean()
prob_score = min(99, max(5, int((avg_corr + 1) * 50)))
c2_trend = "Decrease" if df['Valence'].iloc[-1] < df['Valence'].iloc[-5] else "Increase"

pie_col, trend_col = st.columns([1, 1])

with pie_col:
    fig_pie = go.Figure(go.Pie(values=[prob_score, 100 - prob_score], hole=0.72, showlegend=False, marker=dict(colors=["#0D47A1", "#E6EEF8"])))
    fig_pie.add_annotation(x=0.5, y=0.5, text=f"<b>{prob_score}%</b><br><span style='font-size:13px;'>Bipolar</span>", showarrow=False, font=dict(size=28, color="#0D47A1"))
    fig_pie.update_layout(height=220, margin=dict(l=10, r=10, t=10, b=10), paper_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig_pie, use_container_width=True)

with trend_col:
    trend_arrow = "↗" if c2_trend == "Increase" else "↘"
    st.markdown(f"<div class='kpi-card' style='min-height:220px; text-align:center;'><h3>Tendency</h3><h1 style='font-size:4rem;'>{trend_arrow}</h1><p>{c2_trend}</p></div>", unsafe_allow_html=True)

# --- 3D PLOT ---
st.subheader("3D representation of the patient's emotions on VAD axis")
# Calculs de trajectoire
centroid, direction = get_regression_params(points)
is_outlier = detect_outliers(points, centroid, direction)
line_pts = get_line_visual_points(centroid, direction)

fig = go.Figure()

# 1. Lollipops liées (Têtes + Tiges)
def add_linked_lollipops(pts, dates, mask, color, name, legend_id, symbol='circle'):
    filtered_pts = pts[mask]
    filtered_dates = dates[mask]
    lx, ly, lz = [], [], []
    for p in filtered_pts:
        lx.extend([p[0], p[0], None])
        ly.extend([p[1], p[1], None])
        lz.extend([0, p[2], None])

    fig.add_trace(go.Scatter3d(
        x=lx, y=ly, z=lz, mode='lines',
        line=dict(color=color, width=3),
        showlegend=False, hoverinfo='skip', opacity=0.8, legendgroup=legend_id,
    ))
    fig.add_trace(go.Scatter3d(
        x=filtered_pts[:,0], y=filtered_pts[:,1], z=filtered_pts[:,2],
        mode='markers', name=name, legendgroup=legend_id, customdata=filtered_dates,
        marker=dict(size=8, color=color, symbol=symbol),
        hovertemplate="<b>Date: %{customdata}</b><br> <br>Valence: %{x:.2f}<br>Arousal: %{y:.2f}<br>Dominance: %{z:.2f}<extra></extra>"
    ))

dates_series = pd.to_datetime(df['Date']).dt.strftime('%d/%m/%Y %H:%M')
add_linked_lollipops(points, dates_series, ~is_outlier, 'green', 'inliners', 'Reliable Measurements')
add_linked_lollipops(points, dates_series, is_outlier, 'red', 'outliers','Excluded Measurements', symbol='x')

# 2. Axes Cliniques (BD et Dépression)
fig.add_trace(go.Scatter3d(
    x=[0, 1], y=[0, 1], z=[0, 1],
    mode='lines', name='Bipolar Axis',
    line=dict(color='yellow', width=6),
    hoverinfo='skip'
))

fig.add_trace(go.Scatter3d(
    x=[0, 1], y=[0, 0], z=[0, 0],
    mode='lines', name='Depression Axis',
    line=dict(color='deeppink', width=6),
    hoverinfo='skip'
))

# 3. Ligne de tendance
fig.add_trace(go.Scatter3d(
    x=line_pts[:,0], y=line_pts[:,1], z=line_pts[:,2],
    mode='lines', name='Your overall evolution axis',
    line=dict(color='cyan', width=6),
    hoverinfo='skip'
))

# 4. Passions de Descartes (Repères)
descartes = {                                                                   # dictionnaire des passions selon le modèle de Descartes
        "Love": [1,1,0], "Hate": [0,1,0], "Admiration": [0.5,1,0.5], "Joy": [1,1,0.5], "Sadness": [0,0,0.5], "Desire": [1,0.5,0.5]
    }
for label, p in descartes.items():
    fig.add_trace(go.Scatter3d(
        x=[p[0]], y=[p[1]], z=[p[2]],
        mode='markers+text', name='Descartes Passions',
        marker=dict(size=8, color='purple'),showlegend=False,
        text=[label],                                                           # affichage des noms des émotions
        textposition="top center",
        hovertemplate="<b>%{text}</b><br>Valence: %{x:.2f}<br>Arousal: %{y:.2f}<br>Dominance: %{z:.2f}<extra></extra>"
    ))

fig.update_layout(
    height=800,
    scene=dict(
      # Face X (latérale) -> Transparente
      xaxis=dict(
          range=[-0.02,1.02],
          title="Valence",
          showspikes=False,
          showbackground=False,
      ),
      # Face Y (latérale) -> Transparente
      yaxis=dict(
          range=[-0.02,1.02],
          title="Arousal",
          showspikes=False,
          showbackground=False,
      ),
      # Face Z (sol) -> Pas Transparente
      zaxis=dict(
          range=[-0.02,1.02], #[0,1],
          title="Dominance",
          showbackground=True,
          zeroline=True,
          showspikes=False,
          showgrid=True,
          backgroundcolor="rgba(83, 83, 255, 0.1)"
      ),
      camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2)),                              # position initiale de la caméra (zoom arrière pour avoir tout le cube visible)
      aspectmode='cube'                                                         # garantit que le graphique est un cube parfait et le reste
  ),
  margin=dict(l=0, r=0, b=0, t=50)
)
st.plotly_chart(fig, use_container_width=True)

# --- ÉVOLUTION TEMPORELLE (Projection sur l'axe Bipolaire) ---
st.subheader("Evolution of the projection on the bipolarity axis")

# Vecteur de l'axe bipolaire (1, 1, 1)
bipolar_vector = np.array([1, 1, 1])
bipolar_vector_norm = bipolar_vector / np.linalg.norm(bipolar_vector)

# Calcul de la projection pour chaque point
# (Produit scalaire de chaque ligne par le vecteur unitaire)
df['Score'] = points.dot(bipolar_vector_norm) / np.sqrt(3)

fig_line = go.Figure(go.Scatter(
    x=df['Date'],
    y=df['Score'],
    mode='lines+markers',
    line=dict(color='#FFD700', width=3),
    marker=dict(size=8),
))

fig_line.update_layout(
    height=300,
    margin=dict(l=0,r=0,t=0,b=0),
    yaxis=dict(title="Level (0=Low, 1=High)", range=[0, 1]),
    xaxis=dict(title="Time")
)
st.plotly_chart(fig_line, use_container_width=True)

In [ ]:
# Lancement de Streamlit et du tunnel
# !streamlit run app.py & npx localtunnel --port 8501
!streamlit run app.py \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    --browser.gatherUsageStats false \
    > /dev/null 2>&1 & npx localtunnel --port 8501 | grep "your url is"